In [1]:
GITHUB_REPO_URL = "https://github.com/24520152-ben/LLM-Generated-Multi-Level-Adversarial-Attacks-on-Vietnamese-Hate-Speech-Detection"
BASELINE_OPTIONS = {"model": "svm"}  # Add "epochs": 1 for a test run.

In [2]:
import json
import os
import subprocess
from pathlib import Path

kaggle_working = Path("/kaggle/working")
project_root = Path.cwd()
if kaggle_working.is_dir():
    project_root = kaggle_working / "ie403"
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--", GITHUB_REPO_URL, str(project_root)],
            check=True,
        )
elif not (project_root / "src" / "vihsd_attack").is_dir():
    project_root = project_root.parent

source_dir = project_root / "src"
if not (source_dir / "vihsd_attack").is_dir():
    raise RuntimeError(f"Cannot find src/vihsd_attack in {project_root}.")
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": str(project_root / ".venv"),
    "PYTHONPATH": str(source_dir),
}
if kaggle_working.is_dir() and not env.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.12"],
    cwd=project_root,
    env=env,
    check=True,
)

train_code = """
import json
import sys
from vihsd_attack.classifiers.service import run_baseline
from vihsd_attack.classifiers.config import BaselineConfig

run_baseline(BaselineConfig(**json.loads(sys.argv[1])))
"""
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "-c",
        train_code,
        json.dumps(BASELINE_OPTIONS, default=str),
    ],
    cwd=project_root,
    env=env,
    check=True,
)

Cloning into '/kaggle/working/ie403'...
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 82 packages in 52ms
Prepared 79 packages in 45.62s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 79 packages in 13.45s
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anyio==4.16.0
 + certifi==2026.11.20
 + charset-normalizer==3.5.3
 + click==8.5.1
 + cloudpickle==3.1.3
 + cuda-bindings==13.5.0
 + cuda-pathfinder==1.9.1
 + cuda-toolkit==13.2.2
 + datasets==5.2.0
 + dill==0.4.1
 + filelock==4.3.0
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore2==2.16.2
 + httpx2==2.16.2
 + huggingface-hub==2.4.0
 + idna==3.20
 + jinja2==3.1.6
 + joblib==1.6.0
 + markdown-it-py==4.3.0
 + markupsafe==3.0.4
 + mdurl==0.1.2
 + mpmath==1.3.0
 + multiprocess==0

validation: macro-F1 = 62.59%, accuracy = 86.34%
test: macro-F1 = 61.83%, accuracy = 86.72%
Saved baseline: /kaggle/working/artifacts/baselines/svm/seed-42_20261125T164232Z_9e5325f3


CompletedProcess(args=['uv', 'run', '--no-sync', 'python', '-u', '-c', '\nimport json\nimport sys\nfrom vihsd_attack.classifiers.service import run_baseline\nfrom vihsd_attack.config import BaselineConfig\n\nrun_baseline(BaselineConfig(**json.loads(sys.argv[1])))\n', '{"model": "svm"}'], returncode=0)